# SegOCR — Generate Ensemble Dataset Part B (Kaggle)

Generates **40,000 synthetic samples** (5 worker slices × 8,000 each, ~14.6 GB) at 512², for the second half of the 80K-total split.

Part A covers indices `[0, 40000)`. Part B covers `[40000, 80000)`. Each training worker attaches **both** datasets and symlinks them into one merged directory — so worker N ends up training on 16,000 samples total.

**Before running:**
1. Settings → Accelerator: **None** (CPU is enough).
2. Click **Save Version → Save & Run All**.

**Wall time:** ~1.5–2 hours on Kaggle CPU.

**After it finishes:** click *New Dataset* from the output panel and publish as **`segocr-ensemble-b`** (Public). All 5 training notebooks attach this dataset by that slug (plus the matching part-other dataset).

## 1 / Setup — clone repo + install deps

In [ ]:
import os
if not os.path.isdir('/kaggle/working/segocr'):
    !git clone https://github.com/mauryantitans/SegOCR.git /kaggle/working/segocr
%cd /kaggle/working/segocr
!git pull --quiet
!pip install -q -e .
!pip install -q -r requirements/base.txt

## 2 / Plan + paths

In [ ]:
import os
NUM_WORKERS = 5
SAMPLES_PER_WORKER = 8000   # 5 × 8K = 40K total per dataset (~14.6 GB)
BASE_OFFSET = 40000                  # global index start for part B
DATASET_OUTPUT = '/kaggle/working/segocr-ensemble-b'
os.makedirs(DATASET_OUTPUT, exist_ok=True)

print(f'Part B: 5 × {SAMPLES_PER_WORKER} = {NUM_WORKERS * SAMPLES_PER_WORKER} samples')
print(f'Global index range: [{BASE_OFFSET}, {BASE_OFFSET + NUM_WORKERS * SAMPLES_PER_WORKER})')
print(f'Output: {DATASET_OUTPUT}')

## 3 / Build the generator config

In [ ]:
import yaml
from pathlib import Path

cfg = yaml.safe_load(Path('segocr/configs/default.yaml').read_text())

cfg['generator']['fonts']['root_dir'] = '/usr/share/fonts'
cfg['generator']['fonts']['cache_path'] = '/kaggle/working/font_cache.json'
cfg['generator']['fonts']['min_size'] = 40
cfg['generator']['fonts']['max_size'] = 128
cfg['generator']['image_size'] = [512, 512]
cfg['generator']['num_workers'] = 2

cfg['generator']['text']['min_length'] = 2
cfg['generator']['text']['max_length'] = 20
cfg['generator']['text']['min_words_per_line'] = 1
cfg['generator']['text']['max_words_per_line'] = 3
cfg['generator']['text']['max_lines'] = 1
cfg['generator']['text']['case_distribution'] = {
    'lower': 0.50, 'upper': 0.20, 'mixed': 0.20, 'title': 0.10,
}
cfg['generator']['text']['rare_char_boost'] = 4.0
cfg['generator']['text']['corpus_paths'] = [
    {'path': 'BUNDLED:signs', 'tag': 'signs', 'weight': 0.30},
    {'path': 'BUNDLED:receipts', 'tag': 'receipts', 'weight': 0.20},
    {'path': 'BUNDLED:names', 'tag': 'names', 'weight': 0.20},
    {'path': 'BUNDLED:numbers', 'tag': 'numbers', 'weight': 0.30},
]
cfg['generator']['layout']['modes'] = {
    'horizontal': 0.50, 'rotated': 0.20, 'curved': 0.10,
    'perspective': 0.10, 'deformed': 0.10, 'paragraph': 0.0,
}
cfg['generator']['background']['natural_image_dirs'] = []
cfg['generator']['background']['tier_distribution'] = {
    'tier1_solid': 0.40, 'tier2_procedural': 0.30,
    'tier3_natural': 0.25, 'tier4_adversarial': 0.05,
}
cfg['generator']['compositing']['color_strategy'] = {
    'contrast_aware': 0.60, 'random': 0.30, 'low_contrast': 0.10,
}
cfg['generator']['degradation']['blur'] = {
    'probability': 0.30, 'gaussian_sigma': [0.3, 1.0],
    'motion_kernel': [3, 7], 'defocus_radius': [1, 3],
}
cfg['generator']['degradation']['noise']['probability'] = 0.40
cfg['generator']['degradation']['noise']['gaussian_sigma'] = [5, 20]
cfg['generator']['degradation']['occlusion']['probability'] = 0.05

config_path = '/kaggle/working/gen_config.yaml'
Path(config_path).write_text(yaml.safe_dump(cfg))
print(f'Config: {config_path}')

## 4 / Generate the 5 worker slices for this part

Worker N in this part uses indices `[BASE_OFFSET + N*SAMPLES_PER_WORKER, BASE_OFFSET + (N+1)*SAMPLES_PER_WORKER)`. The deterministic per-index seed guarantees the other gen notebook produces different images for its disjoint range.

In [ ]:
import time
for worker_id in range(NUM_WORKERS):
    output = f'{DATASET_OUTPUT}/worker{worker_id}'
    if os.path.isdir(f'{output}/images') and len(os.listdir(f'{output}/images')) >= SAMPLES_PER_WORKER:
        print(f'[worker {worker_id}] already has {SAMPLES_PER_WORKER} samples — skipping')
        continue
    offset = BASE_OFFSET + worker_id * SAMPLES_PER_WORKER
    end = offset + SAMPLES_PER_WORKER
    print(f'\n=== Generating worker {worker_id}: indices {offset}..{end - 1} ===')
    t0 = time.time()
    !python -m scripts.generate_dataset --config {config_path} --num-images {SAMPLES_PER_WORKER} --output {output} --index-offset {offset}
    print(f'[worker {worker_id}] done in {(time.time() - t0)/60:.1f} min')

print('\nPart B done.')

## 5 / Sanity-check output layout

In [ ]:
for worker_id in range(NUM_WORKERS):
    worker_dir = f'{DATASET_OUTPUT}/worker{worker_id}'
    n_imgs = len(os.listdir(f'{worker_dir}/images')) if os.path.isdir(f'{worker_dir}/images') else 0
    n_sem  = len(os.listdir(f'{worker_dir}/semantic')) if os.path.isdir(f'{worker_dir}/semantic') else 0
    n_inst = len(os.listdir(f'{worker_dir}/instance')) if os.path.isdir(f'{worker_dir}/instance') else 0
    n_meta = len(os.listdir(f'{worker_dir}/metadata')) if os.path.isdir(f'{worker_dir}/metadata') else 0
    print(f'worker{worker_id}: {n_imgs} images / {n_sem} semantic / {n_inst} instance / {n_meta} metadata')

# Total disk usage
!du -sh {DATASET_OUTPUT}

## 6 / Publish as a Kaggle Dataset

1. Click **Save Version → Save & Run All** if you haven't.
2. **Output** tab → **New Dataset**.
3. Slug: **`segocr-ensemble-b`**. Visibility: **Public** (simplest for cross-account sharing).
4. Note the dataset URL — the 5 training notebooks attach this **plus** the other part.